# 简介
- 原学习网站"https://github.com/liaokongVFX/LangChain-Chinese-Getting-Started-Guide/tree/main?tab=readme-ov-file"


# 1. 基础工具使用
## 1.1 搜索查询工具

In [ ]:
import os
from langchain_community.chat_models import ChatTongyi
from langchain_community.utilities import SerpAPIWrapper
from langchain_community.tools import Tool
from langchain_classic import hub
from langchain_classic.agents import create_react_agent, AgentExecutor


# 初始化 Qwen (ChatTongyi)
llm = ChatTongyi(
    model="qwen-max",
    temperature=0,
    max_tokens=2048
)

# 创建搜索工具
search = SerpAPIWrapper()
tools = [
    Tool(
        name="Search",
        func=search.run,
        description="Useful for when you need to answer questions about current events or historical facts. You should ask targeted questions."
    )
]

# 拉取 ReAct 提示模板（来自 LangChain Hub）
prompt = hub.pull("hwchase17/react")

# 创建 ReAct Agent
agent = create_react_agent(llm, tools, prompt)

# 创建 Agent 执行器
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

# 运行查询
response = agent_executor.invoke({
    "input": "What's the date today? What great events have taken place today in history?"
})
print(response["output"])

## 1.2 超长文本理解

In [ ]:
import os
from langchain_community.document_loaders import UnstructuredFileLoader
from langchain_classic.chains.summarize import load_summarize_chain
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.chat_models import ChatTongyi  # 使用 ChatTongyi 替代 OpenAI


# 1. 加载文档
loader = UnstructuredFileLoader("./data/lg_test.txt")
documents = loader.load()
print(f"Original documents: {len(documents)}")

# 2. 分割文本
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0
)
split_documents = text_splitter.split_documents(documents)
print(f"Split into {len(split_documents)} chunks")

# 3. 初始化 Qwen 模型（注意：使用 chat model）
llm = ChatTongyi(
    model="qwen-max",        # 或 qwen-plus / qwen-turbo
    temperature=0,
    max_tokens=1500
)

# 4. 创建总结链（支持 "stuff", "map_reduce", "refine"）
# ⚠️ 注意：refine 对长文本更精细，但对提示词要求高；map_reduce 更稳定
chain = load_summarize_chain(
    llm,
    chain_type="map_reduce",   # 推荐用 map_reduce，兼容性更好
    verbose=True
)

# 5. 执行总结（演示：前5个 chunk）
result = chain.run(split_documents[:5])
print("\n✅ Summary:")
print(result)

# 2. 构建本地知识库问答机器人

In [ ]:
import os
from langchain_community.document_loaders import YoutubeLoader
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_community.chat_models import ChatTongyi
from langchain_classic.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate


# 1. 加载 YouTube 视频字幕
loader = YoutubeLoader.from_youtube_url('https://www.youtube.com/watch?v=Dj60HHy-Kqk', add_video_info=False)
documents = loader.load()
print(f"Loaded {len(documents)} document(s) from YouTube")

# 2. 分割文本
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=20
)
split_docs = text_splitter.split_documents(documents)
print(f"Split into {len(split_docs)} chunks")

# 3. 使用 DashScope Embedding（中文优化）
embeddings = DashScopeEmbeddings(
    model="text-embedding-v2",  # 阿里云官方 embedding 模型
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY")
)

# 4. 构建向量数据库
vector_store = Chroma.from_documents(split_docs, embeddings)
retriever = vector_store.as_retriever()

# 5. 定义对话提示模板（要求用中文回答）
system_template = """
请根据以下上下文回答用户的问题。
如果你不知道答案，请直接说“我不知道”，不要编造内容。
回答必须使用中文。

上下文：
{context}

对话历史：
{chat_history}
"""

# 注意：ConversationalRetrievalChain 默认会传入 {context}, {question}, {chat_history}
messages = [
    SystemMessagePromptTemplate.from_template(system_template),
    HumanMessagePromptTemplate.from_template("{question}")
]
prompt = ChatPromptTemplate.from_messages(messages)

# 6. 初始化 Qwen 聊天模型
llm = ChatTongyi(
    model="qwen-max",
    temperature=0.1,
    max_tokens=2048
)

# 7. 创建对话式检索问答链
qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    condense_question_prompt=prompt,  # 用于生成最终回答的 prompt
    return_source_documents=True,     # 可选：返回参考片段
    verbose=True
)

# 8. 启动交互循环
chat_history = []
print("💬 YouTube 视频问答系统已启动（输入 'quit' 退出）")
while True:
    question = input("\n问题：").strip()
    if question.lower() in ["quit", "exit", "退出"]:
        break
    if not question:
        continue

    try:
        result = qa({"question": question, "chat_history": chat_history})
        answer = result["answer"]
        chat_history.append((question, answer))
        print(f"🤖 答案：{answer}")
        
        # 可选：打印参考来源
        # print("\n📚 参考片段：")
        # for doc in result.get("source_documents", [])[:2]:
        #     print("-", doc.page_content[:150] + "...")
            
    except Exception as e:
        print(f"❌ 出错：{e}")

# 3. 小例子

## 3.1 执行多个chain

In [ ]:
import os
from langchain_community.llms import Tongyi
from langchain_classic.chains import LLMChain
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains import SimpleSequentialChain


# 初始化 Qwen 模型（对应 qwen-max 或其他版本）
llm = Tongyi(
    model_name="qwen-max",      # 也可以用 qwen-plus, qwen-turbo 等
    temperature=1.0,
    max_retries=3,
)

# location 链：根据地点推荐经典菜肴
template = """Your job is to come up with a classic dish from the area that the user suggests.
% USER LOCATION
{user_location}

YOUR RESPONSE:
"""
prompt_template = PromptTemplate(input_variables=["user_location"], template=template)
location_chain = LLMChain(llm=llm, prompt=prompt_template)

# meal 链：根据菜肴名给出简单食谱
template = """Given a meal, give a short and simple recipe on how to make that dish at home.
% MEAL
{user_meal}

YOUR RESPONSE:
"""
prompt_template = PromptTemplate(input_variables=["user_meal"], template=template)
meal_chain = LLMChain(llm=llm, prompt=prompt_template)

# 串联两个链：第一个链的输出作为第二个链的输入
overall_chain = SimpleSequentialChain(chains=[location_chain, meal_chain], verbose=True)

# 运行示例
review = overall_chain.run("Rome")
print(review)

C:\Users\qhjhhh\AppData\Local\Temp\ipykernel_1920\3664715360.py:25: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  location_chain = LLMChain(llm=llm, prompt=prompt_template)
C:\Users\qhjhhh\AppData\Local\Temp\ipykernel_1920\3664715360.py:41: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  review = overall_chain.run("Rome")




> Entering new SimpleSequentialChain chain...
A classic dish from Rome is **Cacio e Pepe**.

This simple yet iconic Roman pasta features only a few ingredients: tonnarelli or spaghetti, Pecorino Romano cheese, freshly ground black pepper, and starchy pasta water. The magic lies in the technique—emulsifying the hot pasta water with finely grated Pecorino to create a creamy, velvety sauce that clings perfectly to the noodles, all elevated by the sharp, salty tang of sheep’s milk cheese and the warmth of cracked pepper.

Originating as a humble meal for shepherds in the surrounding countryside, Cacio e Pepe has become a symbol of Roman culinary tradition—elegant in its simplicity and deeply satisfying in flavor.
**Cacio e Pepe (Simple Home Recipe)**

**Ingredients:**
- 200g spaghetti or tonnarelli  
- 50g Pecorino Romano, finely grated  
- 1–2 tsp freshly ground black pepper  
- Salt (for pasta water)  
- Reserved pasta water (about ½ cup)

**Instructions:**
1. Cook the pasta in a large

## 3.2 结构化输出

In [3]:
import os
from langchain_community.llms import Tongyi
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_classic.prompts import PromptTemplate

# 初始化 Qwen 模型（使用 qwen-max，能力强，适合结构化输出）
llm = Tongyi(
    model_name="qwen-max",
    temperature=0.0,  # 降低随机性，提高格式稳定性
    max_retries=3,
)

# 定义期望的输出字段
response_schemas = [
    ResponseSchema(name="bad_string", description="This is a poorly formatted user input string"),
    ResponseSchema(name="good_string", description="This is your response, a reformatted and corrected version")
]

# 创建结构化解析器
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

# 获取格式指令（通常是 JSON 格式说明）
format_instructions = output_parser.get_format_instructions()

# 构建 Prompt，强调只输出 JSON，不要任何其他内容
template = """
You will be given a poorly formatted string from a user.
Reformat it and correct any spelling mistakes.

{format_instructions}

IMPORTANT: Only output the JSON object. Do not include any other text, explanation, or markdown.

% USER INPUT:
{user_input}

YOUR RESPONSE (JSON ONLY):
"""

prompt = PromptTemplate(
    input_variables=["user_input"],
    partial_variables={"format_instructions": format_instructions},
    template=template
)

# 示例输入
user_input = "welcom to califonya!"
prompt_value = prompt.format(user_input=user_input)

# 调用模型
llm_output = llm.invoke(prompt_value)  # 或直接 llm(prompt_value)

print("Raw LLM Output:")
print(llm_output)
print("\nParsed Output:")

# 尝试解析
try:
    parsed_output = output_parser.parse(llm_output)
    print(parsed_output)
except Exception as e:
    print("Parsing failed:", e)
    print("You may need to use a more robust parser or retry with stricter prompt.")

Raw LLM Output:
```json
{
	"bad_string": "welcom to califonya!",
	"good_string": "Welcome to California!"
}
```

Parsed Output:
{'bad_string': 'welcom to califonya!', 'good_string': 'Welcome to California!'}
